## Eigenshape Analysis with Procrustes Coordinates

This notebook shows the steps to perform an outline analysis with a set of correspondent coordinates. 
The outlines are first standardised with a General Procrustes Analysis (GPA) to retrieve the Procrustes coordinates. These are then used to compute the Procrustes Means between the groups and the Procrustes Distances for each specimen and the between the group means.

### Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
import prince
from SqST_Functions import pca_pd

### Data Preparation

Set the paths needed to the expected input files:
1. The index file is expected to be a data frame with the columns "Index", "Filename" and "Species". Further columns with categories are possible.
2. Outlines provides the CSV with the outlines coordinates for each specimen listed in the index file. Expected columns are "Filename", "x" and "y".
3. The out_path determines the saving location for all generated output files

In [ ]:
outlines =  pd.read_csv('PATH') # Insert path to the table containing outlines coordinates
index = pd.read_csv('PATH') # Insert path to the index file for the dataset
out_path = 'PATH' # Insert path to the designated output folder

Reshape the table of outlines to fit the needed input structure for the General Procrustes Analysis function of the prince module (An array of n_shapes, n_points)

In [ ]:
outlines['coord_id'] = outlines.groupby('Filename').cumcount()
shape_id = np.unique(outlines['Filename'])
n_points  = outlines['coord_id'].nunique()
ol = (outlines.sort_values(['Filename', 'coord_id'])
    .set_index(['Filename', 'coord_id'])[['x', 'x']]
    .to_numpy()
    .reshape(len(shape_id), n_points, 2)
)

### General Procrustes Analysis (GPA)

The GPA standardises the coordinate configurations by eliminating the influence of size, position and orientation.

In [ ]:
GPA_coords = prince.GPA().fit_transform(ol) 

# Save the procrustes coordinates as a DataFrame and CSV
gpa_outlines = (pd.DataFrame(GPA_coords.reshape(len(shape_id) * n_points, 2), columns=['x', 'y'])
    .assign(Filename=np.repeat(shape_id, n_points))[['Filename', 'x', 'y']])

gpa_outlines.to_csv(out_path + 'YOUR_FILENAME.csv', index = False)

### Procrustes Mean 

To retrieve the Procrustes mean shape for the groups, here species, the superimposed coordinates are separated into each group and then the mean is calculated.

In [ ]:
labels = (index.set_index('Filename').loc[shape_id, 'Species'].to_numpy())
all_groups = np.unique(labels)

all_pm = [] # needed later for distance calculation
all_pm_rows = [] # long format table to be saved as a CSV

for group in all_groups:
	# Filter Procrustes coordinates for group:
    idx = np.where(labels == group)[0]
    mean_shape = GPA_coords[idx].mean(axis=0)
    
    # Collect all group means in wide format
    all_pm.append(mean_shape.flatten())

    # Collect all group means for saving
    newname = str(group)
    means = pd.DataFrame({
        'Filename': newname,
	    'x': mean_shape[:, 0],
	    'y': mean_shape[:, 1]
	    })

    all_pm_rows.append(means)

    # Save Procrustes mean for each group
    means.to_csv(out_path + 'PM_' +newname+ '.csv', index=False)

# Create DataFrame for distance calculation
ProcMean_groups = (pd.DataFrame(all_pm, index = all_groups)
    .rename_axis('Species'))

# Create and save the DataFrame with all group means
colGroupMeans = pd.concat(all_pm_rows, ignore_index=True)
colGroupMeans.to_csv(out_path + 'YOUR_FILENAME.csv', index=False)


#### Procrustes Mean of total sample
The mean shape of the complete dataset is calculated as part of the GPA and therefore needs to be extracted from the GPA object create by the prince function

In [ ]:
mean_shape_All = pd.DataFrame(gpa.reference_shape, columns=['x', 'y'])
x = list(mean_shape_All['x'])
y = list(mean_shape_All['y'])

mean_shape_All.to_csv(out_path + 'YOUR_FILENAME.csv')

### Principal Component Analysis (PCA)

The Procrustes coordinates need to be transformed into a wide-format table, with the specimens as rows and the coordinates as the columns to fit the input format of the prince PCA function.

In [ ]:
n_shapes, n_coords, n_dims = GPA_coords.shape
p_coords = GPA_coords.reshape(n_shapes, n_coords * n_dims)
# Define the columns
colums = [f'{c}{i:03d}' for i in range(n_points) for c in ('x', 'y')]
p_coords_df = pd.DataFrame(p_coords, index=shape_id, columns=colums)

The prince.PCA - function allows different settings for the PCA modell, e.g., the number of PCA axes. this has to be defined first and then the PC-scores can be computed.

In [ ]:
pca = prince.PCA(
    n_components= 7,
    rescale_with_std=False,
    engine='sklearn',
    random_state=42).fit(p_coords_df)
    
# Compute PCA
PC_scores = pca.transform(p_coords_df)

# Rename columns and save PC scores
PC_scores.columns = [f'PC{i+1}' for i in range(len(PC_scores.columns))]
PC_scores.to_csv(out_path + 'YOUR_FILENAME.csv', index=True, index_label= 'Filename')

# Save the %variance for the PCA axes
pca.eigenvalues_summary.to_csv(out_path + 'YOUR_FILENAME.csv', index=True)
pca.eigenvalues_summary

#### Extract the Principal Directions for all PCA Axes
For later visualisation of the trajectories of shape change the principal directions for all axes are extracted and saved as a CSV.

In [ ]:
PDs = pca_pd(pca, n_coords, n_pcs=7)
PDs.to_csv(out_path + 'YOUR_FILENAME.csv', index=False)

### Procrustes Distance based on Procrustes Coordinates

The distances between the specimens will be calculated based on the procrustes coordinates and saved as a pairwise matrix

In [ ]:
procDist = pdist(p_coords, metric = 'euclidean')

# Saving the procrustes distances as a pairwise table,
procrustesDist = pd.DataFrame(
	squareform(procDist), # Creating the pairwise matrix
	index=shape_id,
	columns = shape_id)
procrustesDist.index.name='Filename'
procrustesDist.to_csv(out_path + 'YOUR_FILENAME.csv')

### Calculating the Procrustes Distances between the Means of the group

In [ ]:
MeanPdist = pdist(ProcMean_groups, metric = 'euclidean')


# Saving the procrustes distances as a pairwise table,
MeanPdistDF = pd.DataFrame(
	squareform(MeanPdist),
	index=ProcMean_groups.index,
	columns = ProcMean_groups.index)

MeanPdistDF.index.name='Filename'
MeanPdistDF.to_csv(out_path + 'YOUR_FILENAME.csv')